In [1]:
# Make repaired datasets for all california jurisdictions
# SLOW!

import sys
import os
from dotenv import load_dotenv, find_dotenv
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import json

load_dotenv(find_dotenv())

ROOT_PATH = os.getenv("ROOT_PATH")
MY_DATA_PATH = os.getenv("MY_DATA_PATH")
RAW_DATA_PATH = os.getenv("RAW_DATA_PATH")
DEWEY_PATH = os.path.join(RAW_DATA_PATH, "dewey-downloads", "building-permits-united-states")

sys.path.append(os.path.join(ROOT_PATH, "scripts"))
import data_utils as du

sys.path.append(os.path.join(ROOT_PATH, "agent/scripts"))
from data_repair import data_repair, _slugify

SUMMARY_FILEPATH = os.path.join(MY_DATA_PATH, f"dewey_summary.parquet")

COLUMNS = [
    'PERMIT_NUMBER', 'JURISDICTION', 'STATE', 
    'FILE_DATE', 'PERMIT_DATE', 'FINAL_DATE', 
    'STATUS_NORMALIZED', 'STATUS_ORIGINAL', 
    'RECORD_TYPE_ORIGINAL', 'RECORD_SUBTYPE_ORIGINAL', 
    'APN', 'STREET', 'ZIPCODE', 'COUNTY_FIPS', 'CBSA_FIPS',
    'DESCRIPTION', 'DATA'
]  # Columns to load from data file

OUTPUT_COLS = [k for k in COLUMNS if k != 'DATA'] # Don't include DATA in output cols to save space
OUTPUT_COLS = OUTPUT_COLS + ['STATUS_NORMALIZED_FLAG', 'FILE_DATE_FLAG', 'PERMIT_DATE_FLAG', 'FINAL_DATE_FLAG', 'INFERRED_SCHEMA']

OUTPUT_DIR = os.path.join(MY_DATA_PATH, "processed_data")

REPLACE = False   # Whether to replace existing output files


In [2]:
# Load the summary file
summ_df = pd.read_parquet(SUMMARY_FILEPATH)
summ_df = summ_df.loc[summ_df["STATE"] == "CA"]

In [3]:
# Jurisdiction / state
j_df = summ_df[['JURISDICTION', 'STATE']].drop_duplicates()
jurisdictions = j_df['JURISDICTION'].tolist()
states = j_df['STATE'].tolist()

In [4]:
# Iterate through jurisdictions
t0 = time.time()
for i, (jurisdiction, state) in enumerate(zip(jurisdictions, states)):

    print(f"Processing {jurisdiction} {state} ({i+1}/{len(jurisdictions)})")

    jurisdiction_slug = _slugify(jurisdiction)
    state_slug = state.lower().strip()
    os.makedirs(os.path.join(OUTPUT_DIR, f"{state_slug}"), exist_ok=True)
    output_filepath = os.path.join(OUTPUT_DIR, f"{state_slug}", f"{state_slug}_{jurisdiction_slug}_repaired.parquet")

    if os.path.exists(output_filepath) and not REPLACE:
        df = pd.read_parquet(output_filepath)
        if (len(df) > 0) and (set(OUTPUT_COLS).issubset(set(df.columns.tolist()))):
            print(f"    Skipping because output file already exists")
            continue

    files = summ_df.loc[(summ_df["JURISDICTION"] == jurisdiction) & (summ_df["STATE"] == state)]["FILENAME"].tolist()

    city_df = []
    for j, f in enumerate(files):
        dt = (time.time() - t0) / 60
        print(f"\r    Processing file {j+1}/{len(files)} ... elapsed time = {dt:.2f} minutes", end="", flush=True)
        temp_df = pd.read_parquet(os.path.join(DEWEY_PATH, f), columns=COLUMNS)
        temp_df = temp_df.loc[ (temp_df['JURISDICTION'] == jurisdiction) & (temp_df['STATE'] == state)].reset_index(drop=True)
        temp_df = data_repair(temp_df, jurisdiction=jurisdiction, state=state)
        temp_df['FILE_DATE'] = pd.to_datetime(temp_df['FILE_DATE'], errors='coerce', utc=True)
        temp_df['PERMIT_DATE'] = pd.to_datetime(temp_df['PERMIT_DATE'], errors='coerce', utc=True)
        temp_df['FINAL_DATE'] = pd.to_datetime(temp_df['FINAL_DATE'], errors='coerce', utc=True)
        city_df.append(temp_df[OUTPUT_COLS])
    city_df = pd.concat(city_df).reset_index(drop=True)
    city_df.to_parquet(output_filepath)
    print(f"\n    File saved to {output_filepath}.")



Processing Alameda CA (1/250)
    Skipping because output file already exists
Processing Alameda County CA (2/250)
    Skipping because output file already exists
Processing Albany CA (3/250)
    Skipping because output file already exists
Processing Alhambra CA (4/250)
    Skipping because output file already exists
Processing Aliso Viejo CA (5/250)
    Skipping because output file already exists
Processing Anaheim CA (6/250)
    Skipping because output file already exists
Processing Antioch CA (7/250)
    Skipping because output file already exists
Processing Arcadia CA (8/250)
    Skipping because output file already exists
Processing Arroyo Grande CA (9/250)
    Skipping because output file already exists
Processing Arvin CA (10/250)
    Skipping because output file already exists
Processing Atascadero CA (11/250)
    Skipping because output file already exists
Processing Atherton CA (12/250)
    Skipping because output file already exists
Processing Avenal CA (13/250)
    Skipping